# 🛡️ Heritage Shield — YOLOv8 Crack Detector (Colab GPU Trainer)

Trains the **second-layer** YOLOv8 crack detector on the Roboflow `heritage-site-crack-detection` dataset.
This runs on a **free Colab T4 GPU** in ~5–15 minutes.

**Before you start:** `Runtime → Change runtime type → Hardware accelerator → T4 GPU`, then `Save`.

Run each cell top to bottom (Shift+Enter).

## 1. Confirm a GPU is attached
If this prints a table with `Tesla T4` (or similar), you're good. If it errors, set the runtime to GPU as noted above.

In [ ]:
!nvidia-smi

## 2. Install Ultralytics + Roboflow

In [ ]:
!pip -q install ultralytics roboflow
import torch
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())

## 3. Pull the dataset from Roboflow

Get a **free API key**: https://app.roboflow.com → click your account → **Settings → API Keys** → copy the *Private API Key*.
Paste it below. Colab will prompt you securely (the key is not saved in the notebook).

In [ ]:
from getpass import getpass
from roboflow import Roboflow

api_key = getpass('Paste your Roboflow Private API key: ')

rf = Roboflow(api_key=api_key)
project = rf.workspace('ved-waje-uxsri').project('heritage-site-crack-detection')
dataset = project.version(2).download('yolov8')

print('Dataset downloaded to:', dataset.location)

## 4. Train (fast config for GPU)

100 epochs, 640px, with early stopping so it stops once validation plateaus.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')

results = model.train(
    data=f'{dataset.location}/data.yaml',
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,          # early stop if no val improvement for 20 epochs
    name='heritage_crack_yolov8',
    project='runs/detect',
    exist_ok=True,
)

print('Best weights:', results.save_dir)

## 5. Quick validation on the test split (sanity check)

In [ ]:
from ultralytics import YOLO
best = YOLO(f'{results.save_dir}/weights/best.pt')
metrics = best.val(data=f'{dataset.location}/data.yaml', split='test')
print('mAP50:', metrics.box.map50, '| mAP50-95:', metrics.box.map)

## 6. Download `best.pt`

This downloads the trained model as `yolov8_heritage_crack.pt`.
**On your machine, put it here:**
`heritage-shield-backend/data/models/yolov8_heritage_crack.pt`

In [ ]:
import shutil
from google.colab import files

src = f'{results.save_dir}/weights/best.pt'
out = 'yolov8_heritage_crack.pt'
shutil.copy(src, out)
files.download(out)